In [ ]:
from pathlib import Path
import csv
import json
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from rendezvous import (
    U_MAX, INPUT_SET, SEMI_MAJOR_AXIS, sample_state, solve_convex,
    solve_milp, feasible, bisect_horizon, mean_distance,
    plot_validation, plot_grid, plot_monte_carlo, plot_relaxation_examples,
    plot_framework, export_paper_figures,
)

OUTPUT = Path('outputs')
OUTPUT.mkdir(exist_ok=True)
REFERENCE = Path('reference_data')
SOLVER = 'ECOS'  # solver used for the paper; see requirements.txt
PAPER_ECCENTRICITY = 0.7
RUN_MILP = False  # True requires Gurobi and may take several minutes
RUN_GRID_SWEEP = True
RUN_MONTE_CARLO = True
RUN_BISECTION_MONTE_CARLO = False  # supplementary, absent from the paper
print('CVXPY:', cp.__version__, '| installed solvers:', cp.installed_solvers())


## 1. Time horizon and validation

The bisection uses $N=200$, $e=0.7$, a 100–500 s bracket, and a 1 s stopping width. The convex validation uses $N=1000$ at the selected horizon. The mixed-integer linear program (MILP) uses $N=200$ and terminal tolerances of 0.5 m and 0.005 m/s. 


In [ ]:
x0 = np.array([-100., -500., -100., 0., 0., 0.])
selection = bisect_horizon(x0, lower=100., upper=500., tolerance=1.,
                           steps=200, eccentricity=PAPER_ECCENTRICITY, solver=SOLVER)
tf = selection['horizon']
print(f'Feasible upper horizon: {tf:.4f} s; {len(selection["history"]) - 2} midpoint solves')
print(f'Bisection + endpoint checks, wall time: {selection["wall_time"]:.3f} s')
assert abs(tf - 192.1875) < 1., 'Feasibility boundary differs from the recorded run'


In [ ]:
convex = solve_convex(x0, 1000, tf, PAPER_ECCENTRICITY, solver=SOLVER)
assert feasible(convex), convex['status']
print(f'Convex N=1000: cost={convex["cost"]:.6f}, solver={convex["solver_time"]:.4f} s')
print(f'Physical distance={mean_distance(convex["u"], normalized=False):.6g} m/s²; '
      f'normalized distance={mean_distance(convex["u"]):.6g}')

if RUN_MILP:
    milp = solve_milp(x0, 200, tf, PAPER_ECCENTRICITY, terminal_tolerance=1e-3,
                      solver='GUROBI')
else:
    saved = np.load(REFERENCE / 'validation_MILP_N200_CP_N1000_data.npz')
    assert abs(tf - float(saved['tf'])) < 1e-8, 'Cached MILP uses a different horizon'
    assert np.allclose(x0, saved['x0']), 'Cached MILP uses a different initial state'
    milp = {'x': saved['x_milp'], 'u': saved['u_milp'], 'cost': float(saved['milp_cost']),
            'status': str(saved['milp_status']), 'solver_time': float(saved['milp_solver_time']),
            'wall_time': float(saved['milp_wall_time']), 'steps': 200, 'horizon': tf,
            'terminal_residual': float(saved['milp_terminal_residual'])}
print(f'MILP N=200 ({"fresh" if RUN_MILP else "recorded"}): '
      f'cost={milp["cost"]:.6f}, terminal residual={milp["terminal_residual"]:.6g}')
if abs(convex['cost'] - 13.3827) > 0.01:
    print('MANUSCRIPT CHECK: convex cost differs from the 13.3827 currently stated in main_v3.tex.')
print('Different-grid costs should not be interpreted as an exact CP/MILP optimality gap.')
plot_validation(convex, milp, OUTPUT)

np.savez_compressed(OUTPUT / 'validation_solution.npz',
                    tf=tf, x0=x0, x_convex=convex['x'], u_convex=convex['u'],
                    x_milp=milp['x'], u_milp=milp['u'],
                    convex_cost=convex['cost'], milp_cost=milp['cost'])


## 2. Grid-resolution tradeoff

Ten feasible initial states are sampled from $\|r_0\|_\infty\leq500$ m and $\|v_0\|_\infty\leq5$ m/s. For each state, solve at $t_f=200$ s, $e=0.7$, and $N=100,200,\ldots,1000$. A trial state is retained only if it is feasible at $N=100$. The same ten states are used at every grid size.


In [ ]:
GRID_COLUMNS = ['N', 'solver_time_mean_s', 'solver_time_std_s',
                'normalized_distance_mean', 'normalized_distance_std']
if RUN_GRID_SWEEP:
    rng_grid = np.random.RandomState(42)
    states = []
    attempts = 0
    while len(states) < 10 and attempts < 5000:
        attempts += 1
        candidate = sample_state(rng_grid)
        try:
            trial = solve_convex(candidate, 100, 200., PAPER_ECCENTRICITY, solver=SOLVER)
        except cp.SolverError:
            continue
        if feasible(trial):
            states.append(candidate)
    if len(states) < 10:
        raise RuntimeError('Failed to sample 10 feasible initial conditions')
    np.save(OUTPUT / 'grid_initial_states.npy', states)
    grid_rows = []
    for N in range(100, 1001, 100):
        solutions = [solve_convex(x, N, 200., PAPER_ECCENTRICITY, solver=SOLVER) for x in states]
        if not all(map(feasible, solutions)):
            raise RuntimeError(f'Infeasible grid result at N={N}')
        times = np.array([s['solver_time'] for s in solutions])
        distances = np.array([mean_distance(s['u']) for s in solutions])
        grid_rows.append(dict(zip(GRID_COLUMNS, [N, times.mean(), times.std(),
                                                 distances.mean(), distances.std()])))
        print(f'N={N:4d}: {times.mean():.5f} s, normalized distance {distances.mean():.6f}', flush=True)
    with (OUTPUT / 'grid_size_tradeoff.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, GRID_COLUMNS); writer.writeheader(); writer.writerows(grid_rows)
else:
    with (REFERENCE / 'grid_size_tradeoff_mean_std.csv').open(newline='') as f:
        grid_rows = list(csv.DictReader(f))
plot_grid(grid_rows, OUTPUT / 'grid_size_tradeoff_mean_std.pdf')

## 3. Fixed-horizon Monte Carlo experiment and 50-trajectory subset

Sample 1,000 feasible initial conditions. Solve at $t_f=400$ s, $N=400$, and **$e=0.7$**. Record convex solver time (the fixed-horizon solve only), physical distance in m/s², and normalized distance. Save the first 50 accepted trajectories for plotting.


In [ ]:
MC_COLUMNS = ['sample', 'attempt', 'rx', 'ry', 'rz', 'vx', 'vy', 'vz',
              'cost', 'solver_time_s', 'mean_distance', 'normalized_mean_distance']
mc_config = {'eccentricity': PAPER_ECCENTRICITY, 'horizon_s': 400.0,
             'steps': 400, 'samples': 1000, 'trajectory_subset': 50, 'seed': 42}
mc_rows = []
trajectories = []
if RUN_MONTE_CARLO:
    rng_mc = np.random.default_rng(mc_config['seed'])
    attempts = 0
    while len(mc_rows) < mc_config['samples'] and attempts < 5000:
        attempts += 1
        candidate = sample_state(rng_mc)
        try:
            result = solve_convex(candidate, mc_config['steps'], mc_config['horizon_s'],
                                  mc_config['eccentricity'], solver=SOLVER)
        except cp.SolverError:
            continue
        if not feasible(result):
            continue
        mc_rows.append(dict(zip(MC_COLUMNS, [len(mc_rows) + 1, attempts, *candidate,
                                              result['cost'], result['solver_time'],
                                              mean_distance(result['u'], normalized=False),
                                              mean_distance(result['u'])])))
        if len(trajectories) < mc_config['trajectory_subset']:
            trajectories.append(result['x'])
        if len(mc_rows) % 50 == 0:
            print(f'Accepted {len(mc_rows)}/1000 after {attempts} attempts', flush=True)
    if len(mc_rows) != mc_config['samples']:
        raise RuntimeError(f'Only {len(mc_rows)} feasible samples from {attempts} attempts')
    with (OUTPUT / 'monte_carlo_fixed_horizon.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, MC_COLUMNS); writer.writeheader(); writer.writerows(mc_rows)
    np.savez_compressed(OUTPUT / 'trajectory_subset.npz',
                        x=np.stack(trajectories),
                        sample=np.arange(1, mc_config['trajectory_subset'] + 1),
                        x0=np.array([[float(row[key]) for key in ('rx', 'ry', 'rz', 'vx', 'vy', 'vz')]
                                     for row in mc_rows[:mc_config['trajectory_subset']]]))
    (OUTPUT / 'monte_carlo_fixed_horizon_config.json').write_text(
        json.dumps(mc_config, indent=2) + '\n')
else:
    saved_config = json.loads((OUTPUT / 'monte_carlo_fixed_horizon_config.json').read_text())
    if saved_config != mc_config:
        raise RuntimeError('Saved Monte Carlo settings differ from the current manuscript settings')
    with (OUTPUT / 'monte_carlo_fixed_horizon.csv').open(newline='') as f:
        mc_rows = list(csv.DictReader(f))
    if len(mc_rows) != mc_config['samples']:
        raise RuntimeError('The saved Monte Carlo CSV does not contain 1000 accepted cases')
    subset = np.load(OUTPUT / 'trajectory_subset.npz')
    trajectories = list(subset['x'])
    expected_x0 = np.array([[float(row[key]) for key in ('rx', 'ry', 'rz', 'vx', 'vy', 'vz')]
                            for row in mc_rows[:mc_config['trajectory_subset']]])
    if len(trajectories) != 50 or not np.allclose(subset['x0'], expected_x0):
        raise RuntimeError('Saved trajectories are not the first 50 Monte Carlo cases')
for key in ('solver_time_s', 'mean_distance', 'normalized_mean_distance'):
    values = np.array([float(row[key]) for row in mc_rows])
    print(f'{key}: mean={values.mean():.6g}, median={np.median(values):.6g}, max={values.max():.6g}')
plot_monte_carlo(mc_rows, trajectories, OUTPUT)
print(f'Plotted {len(trajectories)} of the 1000 accepted trajectories.')
